# BBC news classification — sample run

This notebook is the narrative counterpart of `run_pipeline.py`. The script is
the canonical, runnable version; this notebook imports it rather than repeating
the code, so the two can never drift apart.

What it shows: every step of the pipeline is *declared* in a config dictionary as
a module and object name, and resolved at run time. Changing the model means
editing a dictionary, not the pipeline.

Fetch the corpus first, from this directory:

```
python download_data.py
```


In [ ]:
import json
from copy import deepcopy
from pathlib import Path

import nltk

from run_pipeline import CONFIG, main

nltk.download('stopwords')
nltk.download('wordnet')


## The pipeline as data

No step is hard coded. Each is a `{module, object}` pair plus its hyper
parameters, resolved through `aistudio.data.meta.module_utils`.


In [ ]:
print(json.dumps(CONFIG, indent=2))


## Running it

`main()` reads the config, resolves each step, trains, prints the classification
report and writes the trained model next to a `report.json` describing the run.


In [ ]:
report = main()


## What the run recorded

The report holds both halves of the story: the config that produced the run and
the numbers that came out of it.


In [ ]:
summary = json.loads(report.toJson(verbose=False))
print('documents :', summary['document_count'])
print('features  :', summary['feature_count'])
print('classes   :', summary['target_names'])
print('accuracy  :', summary['metrics']['accuracy'])


## Swapping a step

To try a different classifier, change two strings. The pipeline code is untouched.

The same applies to the vectorizer, the lemmatizer and the cleaners.


In [ ]:
logistic = deepcopy(CONFIG)
logistic['execute']['model_instance'] = {
    'module': 'sklearn.linear_model',
    'object': 'LogisticRegression',
    'hyper_params': {'max_iter': 1000, 'random_state': 0},
}

logistic_report = main(logistic)


## Where the artefacts land

Each run gets its own timestamped folder under `models/`, holding the persisted
model and the report.


In [ ]:
for path in sorted(Path('models').rglob('*')):
    print(path)
